In [1]:
import os
import glob
import uuid
from typing import List, Dict
from dataclasses import dataclass
import gradio as gr
from dotenv import load_dotenv
import chromadb
from chromadb.config import Settings
from pypdf import PdfReader
from openai import OpenAI
from rich.console import Console
from rich.panel import Panel
from rich.prompt import Prompt

# ==========================
# LOAD ENV VARIABLES
# ==========================
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=OPENAI_API_KEY)

console = Console()

# Ensure data folder exists
os.makedirs("data", exist_ok=True)

d:\Agentic_AI\filesmilitaryRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#DOCUMENT LOADER
class DocumentLoader:
    def __init__(self, chunk_size: int = 800, overlap: int = 150):
        self.chunk_size = chunk_size
        self.overlap = overlap

    def load_all_pdfs(self, folder_path="data") -> List[Dict]:
        documents = []
        pdf_files = glob.glob(os.path.join(folder_path, "*.pdf"))

        if not pdf_files:
            console.print("No PDFs found in data/ folder.", style="red")
            return []

        for pdf_file in pdf_files:
            reader = PdfReader(pdf_file)
            full_text = ""

            for page in reader.pages:
                text = page.extract_text()
                if text:
                    full_text += text + "\n"

            chunks = self._chunk_text(full_text)

            for chunk in chunks:
                documents.append({
                    "id": str(uuid.uuid4()),
                    "text": chunk,
                    "source": os.path.basename(pdf_file)
                })

        return documents

    def _chunk_text(self, text: str) -> List[str]:
        chunks = []
        start = 0

        while start < len(text):
            end = start + self.chunk_size
            chunk = text[start:end]
            chunks.append(chunk)
            start += self.chunk_size - self.overlap

        return chunks

In [3]:
#VECTOR STORE
class VectorStore:
    def __init__(self, persist_directory="chroma_db"):
        self.client = chromadb.Client(
            Settings(
                persist_directory=persist_directory,
                anonymized_telemetry=False
            )
        )

        self.collection = self.client.get_or_create_collection(
            name="army_regulations"
        )
    def embed(self, texts: List[str], batch_size: int = 100):
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

        response = client.embeddings.create(
            model="text-embedding-3-small",
            input=batch
        )

        batch_embeddings = [e.embedding for e in response.data]
        all_embeddings.extend(batch_embeddings)

        return all_embeddings

    def add_documents(self, documents: List[Dict], batch_size: int = 100):
        if not documents:
            return

        for i in range(0, len(documents), batch_size):
            batch = documents[i:i + batch_size]

            texts = [doc["text"] for doc in batch]

            response = client.embeddings.create(
                model="text-embedding-3-small",
                input=texts
        )

            embeddings = [e.embedding for e in response.data]

            self.collection.add(
                ids=[doc["id"] for doc in batch],
                documents=texts,
                embeddings=embeddings,
                metadatas=[{"source": doc["source"]} for doc in batch]
        )



    def search(self, query: str, k: int = 5):
        query_embedding = self.embed([query])[0]

        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=k
        )

        return results["documents"][0]

In [4]:
#LEADING QUESTIONS MODULE
class LeadingQuestions:
    @staticmethod
    def generate(query: str) -> List[str]:
        prompt = f"""
        A soldier asked: "{query}"

        Generate 3 short clarifying follow-up questions 
        relevant to Army regulations and context.
        """

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.4
        )

        questions = response.choices[0].message.content.split("\n")
        return [q.strip("- ").strip() for q in questions if q.strip()]

In [5]:
#RAG CHAIN 
@dataclass
class Message:
    role: str
    content: str


class RAGChain:
    def __init__(self, vector_store: VectorStore):
        self.vector_store = vector_store
        self.memory: List[Message] = []

    def ask(self, query: str):
        retrieved_docs = self.vector_store.search(query)

        context = "\n\n".join(retrieved_docs)

        system_prompt = """You are an expert Army NCO/Officer knowledge assistant with comprehensive knowledge of Army regulations, Field Manuals, and Technical Circulars. Your role is to help junior officers (2LT-CPT) and NCOs (CPL/SPC through 1SG/SGM) with their daily duties.

        CORE RESPONSIBILITIES:
        - Interpret Army regulations, Field Manuals, and Technical Circulars accurately
        - Provide clear, actionable guidance grounded in the provided regulatory context
        - Help leaders understand their responsibilities and the rights of their Soldiers
        - Guide users through administrative and disciplinary processes

        COMMUNICATION STANDARDS:
        - Use military language, rank, and terminology appropriately
        - Be direct and bottom-line-up-front (BLUF) in your responses
        - Structure answers clearly using numbered steps for processes
        - Cite the specific regulation or manual (e.g., "Per AR 670-1...") when giving regulatory guidance
        - Acknowledge when a question requires judgment beyond regulations and recommend consulting the chain of command, JAG, or the appropriate staff section

        IMPORTANT CONSTRAINTS:
        - Base your answers on the regulatory context provided
        - If the provided context does not contain sufficient information, clearly state that and recommend consulting the appropriate staff section or official publication
        - Never provide guidance that contradicts Army policy or the law
        - For SHARP, suicide, or mental health emergencies, always provide crisis resources

        RESPONSE FORMAT:
        - Lead with the direct answer or BLUF
        - Provide regulatory basis when relevant
        - Include action steps where applicable
        - Note any exceptions or important caveats
        - End complex answers with a "Bottom Line" summary

        Context date: {date}
        """

        messages = [{"role": "system", "content": system_prompt}]

        # Add memory
        for msg in self.memory[-5:]:
            messages.append({"role": msg.role, "content": msg.content})

        messages.append({
            "role": "user",
            "content": f"""
            Context:
            {context}

            Question:
            {query}
            """
        })

        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            temperature=0.3
        )

        answer = response.choices[0].message.content

        # Save memory
        self.memory.append(Message("user", query))
        self.memory.append(Message("assistant", answer))

        return answer

In [6]:
vector_store = VectorStore()
rag = RAGChain(vector_store)

response = rag.ask("What does AR 670-1 say about beards?")
print(response)

BLUF: AR 670-1 prohibits beards for Soldiers, except for specific medical or religious accommodations.

1. **General Policy**: Soldiers are not authorized to wear beards while in uniform, as stated in AR 670-1, Chapter 1, Section 3.

2. **Exceptions**: Beards may be permitted for:
   - Medical reasons: Soldiers must provide documentation from a medical professional.
   - Religious accommodations: Soldiers must submit a request through their chain of command, which will be evaluated on a case-by-case basis.

3. **Documentation**: For medical or religious exceptions, ensure that proper documentation is submitted and approved prior to wearing a beard.

Bottom Line: Beards are generally not authorized in the Army, with exceptions for medical or religious reasons subject to approval.


In [7]:


rag = RAGChain(vector_store)

def chat_interface(message, history):
    response = rag.ask(message)
    return response

demo = gr.ChatInterface(chat_interface)
demo.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
